## 실습 개요

실습 목표를 위해 아래 두가지 학습이 필요합니다  

**Supervised Fine-Tuning(SFT)**  
`모델에게 정답을 강제로 보여주면서 학습`

**Direct Preference Optimization(DPO)**  
`사용자 선호도를 반영한 pairwise 학습으로 인간처럼 더 선호되는 출력을 생성`

이번에는 준비된 선호도 데이터셋을 바탕으로 DPO 파인튜닝 전략을 수행해 보겠습니다.

### 학습 전 주의사항 

**GPU 메모리 점검하기**
- 주피터 노트북에서 딥러닝 모델을 로드한 후 커널을 종료하지 않으면, 모델이 GPU 메모리에 남아 있을 가능성이 높습니다.
- 이 상태에서 새로운 모델을 로드하면 메모리 부족으로 에러가 발생할 수 있습니다.
- **주의사항**: 여러 주피터 노트북 파일을 동시에 사용할 경우, 사용하지 않는 노트북의 커널을 종료하여 GPU 메모리를 확보하는게 좋습니다.

## Direct Preference Optimization (DPO)
### 라이브러리 Import

In [ ]:
import os

# 미리 저장된 모델을 사용하기 위해 환경변수 설정
os.environ["HF_HUB_CACHE"] = "/mnt/elice/dataset/hub"

In [ ]:
import torch
from trl import DPOTrainer, DPOConfig
# 모델 및 토크나이저 불러오기 위한 huggingface transformers
from transformers import (
    AutoModelForCausalLM,  # 사전 학습된 언어 모델 로드용
    AutoTokenizer,         # 텍스트를 토큰으로 변환하는 토크나이저
    BitsAndBytesConfig     # 4bit 양자화 설정을 위한 구성 클래스
)

# 학습 효율화를 위한 PEFT (Parameter-Efficient Fine-Tuning) 모듈
from peft import (
    LoraConfig,                       # LoRA 설정 구성
    AutoPeftModelForCausalLM          # 학습된 LoRA 모델 불러오기
)

# 데이터셋 로딩 및 변환용 라이브러리
from datasets import Dataset  # pandas -> huggingface Dataset 변환용
import pandas as pd           # CSV 파일 로딩 등 데이터 처리

#### 1. 데이터 로드 및 포맷

In [ ]:
# 선호 데이터셋 로드
# prompt, chosen, rejected 형식의 CSV 파일 로드
pref_df = pd.read_csv("data/processed/preference_dataset.csv")

# huggingface Dataset 형식으로 변환
pref_dataset = Dataset.from_pandas(pref_df)

# DPOTrainer가 요구하는 입력 형태로 맵핑(불필요한 컬럼 제거)
def return_prompt_and_responses(samples):
    return {
        "prompt": samples['prompt'],
        "chosen": samples['chosen'],
        "rejected": samples['rejected']
    }


pref_dataset = pref_dataset.map(
    return_prompt_and_responses,
    remove_columns=pref_dataset.column_names
)

# 데이터 확인
print(type(pref_dataset )) # 데이터셋 정보
print('---------------------------------------------------------------------')
print(pref_dataset)
print('---------------------------------------------------------------------')
print('prompt :',pref_dataset[4]['prompt'])
print('---------------------------------------------------------------------')
print('chosen',pref_dataset[4]['chosen'])
print('---------------------------------------------------------------------')
print('rejected',pref_dataset[4]['rejected'])

#### 2. 토크나이저 및 모델 로드 (SFT 모델)

In [ ]:
# 토크나이저 로드 및 설정
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 해당 모델에 맞는 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 패딩 토큰 수동 설정 (Llama는 기본 pad token이 없음)
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.pad_token = "<|pad|>"
tokenizer.padding_side = "right"  # 오른쪽 패딩으로 설정

# LoRA adapter만 포함된 모델 불러오기
model = AutoPeftModelForCausalLM.from_pretrained(
    "final_adapter",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
# 참조 모델 없이 학습하는 경우에도 캐시 비활성화 필요
model.config.use_cache = False

#### 3. LoRA 설정 (DPO용)

In [ ]:
peft_config = LoraConfig(
    r=16,                         # 랭크(rank) 값: LoRA 행렬의 저차원 공간 차원 수
    lora_alpha=16,               # LoRA scaling factor: 실제 학습 시 gradient에 곱해지는 값 (출력 스케일 조정)
    lora_dropout=0.1,            # 학습 중 dropout 적용 확률 (과적합 방지용)
    task_type="CAUSAL_LM",       # 모델 타입: AutoModelForCausalLM 기반 모델이라는 뜻
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]  
    # LoRA를 적용할 대상 모듈 이름들 (Llama 모델의 self-attention projection 계층)
)

#### 4. DPO Trainer 구성

In [ ]:
# DPO 학습 인자 설정
training_args = DPOConfig(
    output_dir="outputs/dpo_model",             # 학습 결과 및 체크포인트 저장 디렉토리
    per_device_train_batch_size=1,              # GPU 1개당 배치 크기
    gradient_accumulation_steps=1,              # 작은 배치 크기를 보완하기 위한 gradient 누적 step
    gradient_checkpointing=True,                # 메모리 효율을 위한 gradient checkpointing 사용
    gradient_checkpointing_kwargs={'use_reentrant': False},  # 새로운 방식의 체크포인팅
    num_train_epochs=2,                         # 전체 학습 데이터셋에 대해 2번 반복 학습
    max_grad_norm=0.3,                          # gradient clipping 임계값 (너무 큰 gradient를 제한)
    learning_rate=1e-6,                         # 학습률 (DPO에서는 일반적으로 작게 설정)
    bf16=True,                                  # bfloat16 포맷 사용 (AMP보다 안정적이며 메모리 절약)
    logging_steps=10,                           # 로그 출력 주기 (10 step마다 loss 등 출력)
    optim="paged_adamw_32bit",                  # 양자화된 모델을 위한 효율적인 옵티마이저
    lr_scheduler_type="cosine",                 # cosine 스케줄러로 학습률 점진 감소
    warmup_ratio=0.05,                          # 전체 학습 step 중 5%를 워밍업으로 사용
    remove_unused_columns=True,                 # 불필요한 feature 제거
    max_prompt_length=512,                      # 프롬프트의 최대 길이
    max_length=1024,                            # 프롬프트 + 응답 포함한 전체 입력 길이 제한
    save_strategy="epoch",                      # 에폭 단위로 체크포인트 저장
    beta=0.2                                    # DPO의 중요 하이퍼파라미터 (선호와 비선호 차이를 강조)
)

#  DPOTrainer 초기화
trainer = DPOTrainer(
    model=model,                 # LoRA가 적용된 초기 base 모델
    ref_model=None,             # 참조 모델 생략 (implicit 방식 사용)
    args=training_args,         # 위에서 정의한 학습 인자
    train_dataset=pref_dataset, # DPO 학습용 데이터셋: (prompt, chosen, rejected) 구조 필요
    peft_config=peft_config,    # LoRA 설정 (적용 계층, rank 등)
    processing_class=tokenizer  # 프롬프트 전처리에 사용될 tokenizer
)

#### 5. DPO 학습 수행

Note. 학습에는 약 1시간 정도 소요됩니다. 학습 결과가 미리 준비되어 있으므로, 이 셀의 실행을 강제로 끊고, 다음 셀로 넘어가셔도 무방합니다.

In [ ]:
# 학습 시작
trainer.train() # 모델 학습 수행

# 어댑터 저장
adapter_dir = "outputs/dpo_model/final_adapter"
trainer.save_model(adapter_dir) # LoRA adapter 가중치 저장
tokenizer.save_pretrained(adapter_dir) # tokenizer도 함께 저장 (추론 재현을 위해)

#################################################################################

# LoRA + base 모델 병합 및 저장
# 학습된 LoRA 모델 로드 후 병합
final_dpo = AutoPeftModelForCausalLM.from_pretrained(
    adapter_dir,
    device_map="cpu",
    torch_dtype=torch.bfloat16
)

# LoRA 가중치를 base 모델에 병합 (LoRA 제거, 순수한 합성 모델로 변환)
final_dpo = final_dpo.merge_and_unload()

# 병합된 모델 저장 (추론용)
output_path = "outputs/dpo_model/merged"
final_dpo.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

드디어 학습이 끝났습니다!  
다음 장에서 Vanilla 모델과 SFT 모델, DPO 모델의 출력들을 비교해보겠습니다.

### 실습 : 학습이 끝난 모델을 로드하여 추론하기

아래 요구사항을 참고하여, 각 요구사항에 대응하는 코드의 None을 적절한 코드로 대체하여 학습이 끝난 모델을 로드하여 추론하는 코드를 완성하세요. 

요구사항 1 : meta-llama/Llama-3.2-1B-Instruct 모델을 vanilla_model로 불러오세요.     
요구사항 2 :  
 - DPO로 학습된 LoRA 어댑터가 저장된 디렉토리는 "dpo_model/checkpoint-10556"입니다.
 - 이 어댑터를 base 모델에 적용하여 최종 모델을 구성하세요. 
  
요구사항 3 :  
 - LLaMA 3 Instruct 포맷에 따라 프롬프트를 구성하세요:
 - system/user/assistant 구조를 따르되, <|begin_of_text|>, <|eot_id|> 등도 포함할 것
  
요구사항 4 : 사용자의 질문은 input() 함수로 받도록 하세요.  
요구사항 5 : 실시간 출력이 되도록 TextStreamer를 사용하여 결과를 출력하세요.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from peft import PeftModel

# 공통 설정
max_new_tokens = 300

# LLaMA 3 스타일 프롬프트 템플릿
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id>
{}\n<|eot_id|><|start_header_id|>user<|end_header_id>
{}\n<|eot_id|><|start_header_id|>assistant<|end_header_id>
"""

# Step 1. 사용자 질문 입력
user_prompt = None # 요구사항 4

# Step 2. 프롬프트 생성
system_prompt = "간결하고 명확한 한국어로 질문에 답변하세요."
full_prompt = prompt_template.format(system_prompt, user_prompt)

# Step 3. 프롬프트 디버깅 출력
print("Generated Prompt:\n", full_prompt)

# Step 4. tokenizer 로드 (LoRA adapter와 함께 저장된 디렉토리)
adapter_path = None # 요구사항 2-1
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Step 5. base 모델 로드
vanilla_model = None # 요구사항 1

vanilla_model.resize_token_embeddings(len(tokenizer))
# Step 6. LoRA 어댑터 적용
sft_model = PeftModel.from_pretrained(None) # 요구사항 2-2

# Step 7. streamer 설정 (디버깅용으로 prompt 및 특수 토큰 출력)
streamer = TextStreamer(tokenizer, skip_special_tokens=False, skip_prompt=False)

# Step 8. 추론 및 실시간 출력
inputs = tokenizer(full_prompt, return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    sft_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.5,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        # 요구사항 5
    )